In [385]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder

In [386]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

In [389]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [387]:
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# <b>Load Dataset</b>

In [421]:
df = pd.read_csv('data/0611raw_final.csv') # 이미지(밝기, 명도 안 들어감)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483 entries, 0 to 482
Columns: 149 entries, Unnamed: 0 to rewards_count
dtypes: float64(56), int64(84), object(9)
memory usage: 562.4+ KB


In [423]:
df_con = pd.read_csv('data/contrast.csv') 

In [425]:
df = df.merge(
    df_con[[
        "projectID",
        "edge_density",
        "saturation",
        "brightness",
        "contrast"
    ]],
    on="projectID",
    how="left"
)

In [432]:
post_cols = [
    "fundedInSeconds", "likes_0", "likes_1", "is_backer_0", "is_backer_1"
]

text_cols = [
    "creator_id_0", "creator_id_1",
    "is_pledge_master_0", "is_pledge_master_1",
    
    "is_prior_backer_0", "is_prior_backer_1",
    "is_pathfinder_0", "is_pathfinder_1",
     "is_normal_0", "is_normal_1"
]

text_cols_api = [
    "Product_Question_count_0", "Product_Question_count_1",
    "Suggestion_Idea_count_0", "Suggestion_Idea_count_1",
    "Praise_Support_count_0", "Praise_Support_count_1",
    "Shipping_Fulfillment_count_0", "Shipping_Fulfillment_count_1",
    "Complaint_Refund_count_0", "Complaint_Refund_count_1",
    "Spam_Irrelevant_count_0", "Spam_Irrelevant_count_1"
]

text_cols_sen = [
    "긍정_0", "긍정_1",
    "부정_0", "부정_1",
    "중립_0", "중립_1"
    ]

text_cols_topic = [
    "Campaign / Pledge_0", "Campaign / Pledge_1",
    "Community Reaction_0", "Community Reaction_1",
    "Components & Production_0", "Components & Production_1",
    "Gameplay / Rules / Content_0", "Gameplay / Rules / Content_1",
    "Language & Localization_0", "Language & Localization_1",
    "Schedule / Updates_0", "Schedule / Updates_1",
    "Shipping / Price / Tax_0", "Shipping / Price / Tax_1"
]

one_month_cols = [
    "campaignGoal_usd_1m",
    "fundsGathered_usd_1m",
    "price_usd_1m"
]

six_month_cols = [
    "campaignGoal_usd_6m",
    "fundsGathered_usd_6m",
    "price_usd_6m"
]

color_cols1 = [
    "R1", "G1", "B1",
    "R2", "G2", "B2",
    "R3", "G3", "B3",
    "R4", "G4", "B4"]
color_cols2= [
    "ks_color_1", "ks_color_2", "ks_color_3", "ks_color_4"] # mae줄음
color_cols3=[
    "emotion_adjective_1", "emotion_adjective_2",
    "emotion_adjective_3", "emotion_adjective_4"
]

edge_cols1 = ["edge_density"]
edge_cols2 = ["saturation"]
edge_cols3 = ["brightness"]
edge_cols4 = ["contrast"]

In [433]:
# drop_cols = post_cols + text_cols_api + text_cols_sen + text_cols_topic + one_month_cols + color_cols
drop_cols = (post_cols + six_month_cols
             + color_cols1 + color_cols3
             + edge_cols1 + edge_cols2 + edge_cols3 + edge_cols4 
             + text_cols_topic + text_cols_api)# 고정


# drop_cols = (post_cols + six_month_cols
             # + color_cols1 + color_cols3
             # + edge_cols1 + edge_cols2 + edge_cols3 + edge_cols4 
             # + text_cols_topic)# 고정
            

df.drop(columns=drop_cols, errors="ignore", inplace=True)

# <b>이상치 제거</b>

In [436]:
# ======================================================================
# 1.5 IQR 기준으로 fundsGathered_usd_1m 상방 이상치 제거
# ======================================================================

col_target = "fundsGathered_usd_1m"

# Q1, Q3, IQR 계산
q1 = df[col_target].quantile(0.25)
q3 = df[col_target].quantile(0.75)
iqr = q3 - q1

# 상한선 계산
upper_bound = q3 + 1.5 * iqr

# 상한선 이하 데이터만 남기기
df = df.loc[df[col_target] <= upper_bound].reset_index(drop=True)

print("📊 [IQR 기준 이상치 제거 완료]")
print(f"Q1: ${q1:,.2f}")
print(f"Q3: ${q3:,.2f}")
print(f"IQR: ${iqr:,.2f}")
print(f"IQR 상한선: ${upper_bound:,.2f}")
print(f"남은 데이터 크기: {df.shape}")

📊 [IQR 기준 이상치 제거 완료]
Q1: $3,989.53
Q3: $224,294.09
IQR: $220,304.56
IQR 상한선: $554,750.94
남은 데이터 크기: (418, 98)


# <b>train test split</b>

In [440]:
from sklearn.model_selection import train_test_split

df_trn, df_tst = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

# <b>EDA</b>

In [84]:
df_trn.info()

<class 'pandas.core.frame.DataFrame'>
Index: 334 entries, 336 to 102
Data columns (total 88 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   projectID                  334 non-null    int64  
 1   currencySymbol             334 non-null    object 
 2   enableBoardGameProperties  334 non-null    int64  
 3   minPlayers                 323 non-null    float64
 4   maxPlayers                 323 non-null    float64
 5   minAge                     314 non-null    float64
 6   playTime                   317 non-null    float64
 7   isDiscounted               334 non-null    int64  
 8   previous_campaigns_count   334 non-null    int64  
 9   duration_days              334 non-null    int64  
 10  softclose                  334 non-null    int64  
 11  campaignGoal_usd_1m        334 non-null    float64
 12  fundsGathered_usd_1m       334 non-null    float64
 13  price_usd_1m               334 non-null    float64
 1

In [85]:
df_tst.info()

<class 'pandas.core.frame.DataFrame'>
Index: 84 entries, 321 to 66
Data columns (total 88 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   projectID                  84 non-null     int64  
 1   currencySymbol             84 non-null     object 
 2   enableBoardGameProperties  84 non-null     int64  
 3   minPlayers                 81 non-null     float64
 4   maxPlayers                 81 non-null     float64
 5   minAge                     79 non-null     float64
 6   playTime                   79 non-null     float64
 7   isDiscounted               84 non-null     int64  
 8   previous_campaigns_count   84 non-null     int64  
 9   duration_days              84 non-null     int64  
 10  softclose                  84 non-null     int64  
 11  campaignGoal_usd_1m        84 non-null     float64
 12  fundsGathered_usd_1m       84 non-null     float64
 13  price_usd_1m               84 non-null     float64
 14 

In [86]:
df.columns

Index(['projectID', 'currencySymbol', 'enableBoardGameProperties',
       'minPlayers', 'maxPlayers', 'minAge', 'playTime', 'isDiscounted',
       'previous_campaigns_count', 'duration_days', 'softclose',
       'campaignGoal_usd_1m', 'fundsGathered_usd_1m', 'price_usd_1m',
       'creator_id_0', 'creator_id_1', 'is_pledge_master_0',
       'is_pledge_master_1', 'is_prior_backer_0', 'is_prior_backer_1',
       'is_pathfinder_0', 'is_pathfinder_1', 'is_normal_0', 'is_normal_1',
       '4X', 'AR Next', 'ARNext26', 'Action', 'Adventure', 'Area Control',
       'Asymmetric', 'Campaign', 'Card Game', 'Cats', 'Civilization',
       'Collectible', 'Collectible Models', 'Competitive', 'Cooperative',
       'Deck Building', 'Deduction', 'Dexterity', 'Dice Game', 'Digital',
       'Economic', 'Educational', 'Exploration', 'Family', 'Fantasy', 'Feast',
       'First-Timers', 'Game Components', 'History', 'Horror', 'Humor',
       'Legacy', 'Logical', 'MOBA', 'Major Creators', 'Media', 'Modern',
 

In [ ]:
import matplotlib.pyplot as plt
import math

# 한글 폰트 깨짐 해결 - 윈도우 기준
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

tag_cols = [
    "4X", "Action", "Adventure",
    "Area Control", "Asymmetric", "Campaign", "Card Game", "Cats",
    "Civilization", "Collectible", "Collectible Models", "Competitive",
    "Cooperative", "Deck Building", "Deduction", "Dexterity", "Dice Game",
    "Digital", "Economic", "Educational", "Exploration", "Family",
    "Fantasy", "Feast", "First-Timers", "Game Components", "History",
    "Horror", "Humor", "Legacy", "Logical", "MOBA", 
    "Media", "Modern", "Multiplayer", "Mythology", "Narrative",
    "OctoberFeast25", "Paint", "Party game", "Political",
    "Print and Play", "RPG", "Racing", "Reprints",
    "Resource management", "Science Fiction", "Set Collection", "Sport",
    "Strategy", "Survival", "TTRPG", "Terrain Building", "Tower Defense",
    "Video Game", "Video Game Theme", "Wargame", "WinterFeast2026",
    "Worker placement", "dinotuesday"
]

drop_cols_eda = color_cols + tag_cols

# tag 컬럼 + color 컬럼 빼기
df_trn_no = df_trn.drop(columns=drop_cols_eda, errors="ignore")

# 숫자형 컬럼만 선택
num_cols = df_trn_no.select_dtypes(include="number").columns

# 한 행에 4개씩
n_cols = 4
n_rows = math.ceil(len(num_cols) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(20, n_rows * 3)
)

axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df_trn_no[col].dropna(), bins=30)
    axes[i].set_title(col, fontsize=10)
    axes[i].grid(True)

# 남는 빈 그래프 삭제
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

# 한글 폰트 깨짐 해결 - 윈도우 기준
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

tag_cols = [
    "4X", "Action", "Adventure",
    "Area Control", "Asymmetric", "Campaign", "Card Game", "Cats",
    "Civilization", "Collectible", "Collectible Models", "Competitive",
    "Cooperative", "Deck Building", "Deduction", "Dexterity", "Dice Game",
    "Digital", "Economic", "Educational", "Exploration", "Family",
    "Fantasy", "Feast", "First-Timers", "Game Components", "History",
    "Horror", "Humor", "Legacy", "Logical", "MOBA", 
    "Media", "Modern", "Multiplayer", "Mythology", "Narrative",
    "OctoberFeast25", "Paint", "Party game", "Political",
    "Print and Play", "RPG", "Racing", "Reprints",
    "Resource management", "Science Fiction", "Set Collection", "Sport",
    "Strategy", "Survival", "TTRPG", "Terrain Building", "Tower Defense",
    "Video Game", "Video Game Theme", "Wargame", "WinterFeast2026",
    "Worker placement", "dinotuesday"
]

drop_cols_eda = color_cols + tag_cols

# tag 컬럼 + color 컬럼 빼기
df_tst_no = df_tst.drop(columns=drop_cols_eda, errors="ignore")

# 숫자형 컬럼만 선택
num_cols = df_tst_no.select_dtypes(include="number").columns

# 한 행에 4개씩
n_cols = 4
n_rows = math.ceil(len(num_cols) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(20, n_rows * 3)
)

axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df_tst_no[col].dropna(), bins=30)
    axes[i].set_title(col, fontsize=10)
    axes[i].grid(True)

# 남는 빈 그래프 삭제
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# <b>Preprocessing</b>

In [248]:
dummy_cols = [
    "4X",
    "AR Next",
    "ARNext26",
    "Action",
    "Adventure",
    "Area Control",
    "Asymmetric",
    "Campaign",
    "Card Game",
    "Cats",
    "Civilization",
    "Collectible",
    "Collectible Models",
    "Competitive",
    "Cooperative",
    "Deck Building",
    "Deduction",
    "Dexterity",
    "Dice Game",
    "Digital",
    "Economic",
    "Educational",
    "Exploration",
    "Family",
    "Fantasy",
    "Feast",
    "First-Timers",
    "Game Components",
    "History",
    "Horror",
    "Humor",
    "Legacy",
    "Logical",
    "MOBA",
    "Major Creators",
    "Media",
    "Modern",
    "Multiplayer",
    "Mythology",
    "Narrative",
    "OctoberFeast25",
    "Paint",
    "Party game",
    "Political",
    "Print and Play",
    "RPG",
    "Racing",
    "Reprints",
    "Resource management",
    "Science Fiction",
    "Set Collection",
    "Sport",
    "Strategy",
    "Survival",
    "TTRPG",
    "Terrain Building",
    "Tower Defense",
    "Video Game",
    "Video Game Theme",
    "Wargame",
    "WinterFeast2026",
    "Worker placement",
    "dinotuesday"
]

In [ ]:
df_trn[dummy_cols].sum().sort_values(ascending=False)

In [ ]:
df_tst[dummy_cols].sum().sort_values(ascending=False)

In [447]:
df_trn.drop(columns=['AR Next', 'ARNext26', 'Major Creators'], inplace=True)
df_tst.drop(columns=['AR Next', 'ARNext26', 'Major Creators'], inplace=True)

In [444]:
tag_cols = [
    "4X", "Action", "Adventure",
    "Area Control", "Asymmetric", "Campaign", "Card Game", "Cats",
    "Civilization", "Collectible", "Collectible Models", "Competitive",
    "Cooperative", "Deck Building", "Deduction", "Dexterity", "Dice Game",
    "Digital", "Economic", "Educational", "Exploration", "Family",
    "Fantasy", "Feast", "First-Timers", "Game Components", "History",
    "Horror", "Humor", "Legacy", "Logical", "MOBA", 
    "Media", "Modern", "Multiplayer", "Mythology", "Narrative",
    "OctoberFeast25", "Paint", "Party game", "Political",
    "Print and Play", "RPG", "Racing", "Reprints",
    "Resource management", "Science Fiction", "Set Collection", "Sport",
    "Strategy", "Survival", "TTRPG", "Terrain Building", "Tower Defense",
    "Video Game", "Video Game Theme", "Wargame", "WinterFeast2026",
    "Worker placement", "dinotuesday"
]

In [ ]:
genre_cols = tag_cols

In [446]:
target = "fundsGathered_usd_1m"

genre_y_summary = []

for col in genre_cols:
    temp = df_trn.groupby(col)[target].agg(["count", "mean", "median", "std"])
    temp["genre"] = col
    
    if 1 in temp.index:
        genre_y_summary.append(temp.loc[1])

genre_y_summary = pd.DataFrame(genre_y_summary)

genre_y_summary = genre_y_summary[["genre", "count", "mean", "median", "std"]]

# count > 10인 장르만 남기기
genre_y_summary = genre_y_summary[genre_y_summary["count"] >= 10].copy()

# median이 전체 y 분포에서 상위 몇 %인지
genre_y_summary["median_top_pct"] = genre_y_summary["median"].apply(
    lambda x: (df_trn[target] >= x).mean() * 100
).round(2)

# median 높은 순 정렬 + 인덱스 초기화
genre_y_summary = genre_y_summary.sort_values("median", ascending=False).reset_index(drop=True)

# 순위 열 추가
genre_y_summary["rank"] = genre_y_summary.index + 1
genre_y_summary["rank"] = genre_y_summary["rank"].apply(lambda x: str(x) + "위")

# 열 순서 정리
genre_y_summary = genre_y_summary[
    ["rank", "genre", "count", "mean", "median", "std", "median_top_pct"]
]

genre_y_summary

,rank,genre,count,mean,median,std,median_top_pct
0,1위,Narrative,22,163921.86,128780.09,172814.29,20.06
1,2위,Collectible Models,14,140181.42,96152.52,143742.73,23.95
2,3위,Video Game Theme,10,154824.65,64442.29,178865.16,32.34
3,4위,Worker placement,24,148610.71,56152.86,177708.54,34.73
4,5위,Terrain Building,11,74896.47,50202.71,90292.36,37.43
5,6위,Cooperative,73,128392.04,50202.71,150321.24,37.43
6,7위,Exploration,44,121547.42,48282.33,154470.08,38.32
7,8위,Horror,35,110013.67,46984.72,155944.23,38.92
8,9위,Area Control,35,136756.15,45765.07,175568.52,40.12
9,10위,Adventure,78,118050.87,44318.17,148944.50,40.72


# <b>결측치</b>

In [448]:
cols = ["minPlayers", "maxPlayers", "minAge", "playTime"]

for col in cols:
    med = df_trn[col].median()

    df_trn[col] = df_trn[col].fillna(med)
    df_tst[col] = df_tst[col].fillna(med)

# <b>X Y split</b>

In [450]:
total_cols = [x for x in df_trn.columns]

id_cols = ['projectID']
y_cols = ['fundsGathered_usd_1m']
x_cols = [col for col in total_cols if col not in id_cols+y_cols]

df_trn_x = df_trn[x_cols]
df_trn_y = df_trn[y_cols]

# df_tst.drop(columns='fundsGathered_usd_6m', inplace=True)
df_tst_x = df_tst[x_cols]
df_tst_y = df_tst[y_cols]

# <b>인코딩 & 피처 엔지니어링</b>

In [453]:
df_trn_x = pd.get_dummies(
    df_trn_x,
    columns=["currencySymbol"],
    prefix="currencySymbol",
    drop_first=False
)

df_tst_x = pd.get_dummies(
    df_tst_x,
    columns=["currencySymbol"],
    prefix="currencySymbol",
    drop_first=False
)

In [456]:
def genre_performance_group(top_pct):
    if top_pct < 31:
        return "top_30%"
    elif top_pct <= 50:
        return "top_30%_50%"
    elif top_pct <= 70:
        return "top_50%_70%"
    else:
        return "bottom_30%"

genre_y_summary["genre_perf_group"] = genre_y_summary["median_top_pct"].apply(
    genre_performance_group
)

genre_y_summary.head(5)

,rank,genre,count,mean,median,std,median_top_pct,genre_perf_group
0,1위,Narrative,22,163921.86,128780.09,172814.29,20.06,top_30%
1,2위,Collectible Models,14,140181.42,96152.52,143742.73,23.95,top_30%
2,3위,Video Game Theme,10,154824.65,64442.29,178865.16,32.34,top_30%_50%
3,4위,Worker placement,24,148610.71,56152.86,177708.54,34.73,top_30%_50%
4,5위,Terrain Building,11,74896.47,50202.71,90292.36,37.43,top_30%_50%


In [457]:
top_30_genres = genre_y_summary.loc[
    genre_y_summary["genre_perf_group"] == "top_30%",
    "genre"
].tolist()

top_30_50_genres = genre_y_summary.loc[
    genre_y_summary["genre_perf_group"] == "top_30%_50%",
    "genre"
].tolist()

top_50_70_genres = genre_y_summary.loc[
    genre_y_summary["genre_perf_group"] == "top_50%_70%",
    "genre"
].tolist()

bottom_30_genres = genre_y_summary.loc[
    genre_y_summary["genre_perf_group"] == "bottom_30%",
    "genre"
].tolist()

print("top_30_genres:", top_30_genres)
print("top_30_50_genres:", top_30_50_genres)
print("top_50_70_genres:", top_50_70_genres)
print("bottom_30_genres:", bottom_30_genres)


top_30_genres: ['Narrative', 'Collectible Models']
top_30_50_genres: ['Video Game Theme', 'Worker placement', 'Terrain Building', 'Cooperative', 'Exploration', 'Horror', 'Area Control', 'Adventure', 'Asymmetric', 'Game Components', 'Economic', 'Campaign', 'Science Fiction', 'Action', 'History', 'Wargame', 'Deck Building', 'Fantasy', 'Set Collection', 'Resource management', 'Competitive', 'Mythology', 'Strategy']
top_50_70_genres: ['Modern', 'Sport', 'Deduction', 'RPG', 'Multiplayer', 'Survival', 'Political', 'Racing', 'Card Game', 'Dice Game', 'Educational']
bottom_30_genres: ['Family', 'Logical', 'Humor', 'Collectible', 'Party game', 'Digital']


In [458]:
df_trn_x["top_30_genre_count"] = df_trn_x[top_30_genres].sum(axis=1)
df_tst_x["top_30_genre_count"] = df_tst_x[top_30_genres].sum(axis=1)

df_trn_x["top_30_50_genre_count"] = df_trn_x[top_30_50_genres].sum(axis=1)
df_tst_x["top_30_50_genre_count"] = df_tst_x[top_30_50_genres].sum(axis=1)

df_trn_x["top_50_70_genre_count"] = df_trn_x[top_50_70_genres].sum(axis=1)
df_tst_x["top_50_70_genre_count"] = df_tst_x[top_50_70_genres].sum(axis=1)

df_trn_x["bottom_30_genre_count"] = df_trn_x[bottom_30_genres].sum(axis=1)
df_tst_x["bottom_30_genre_count"] = df_tst_x[bottom_30_genres].sum(axis=1)


In [459]:
df_trn_x["total_genre_count"] = df_trn_x[genre_cols].sum(axis=1)
df_tst_x["total_genre_count"] = df_tst_x[genre_cols].sum(axis=1)

df_trn_x["top_30_genre_ratio"] = (
    df_trn_x["top_30_genre_count"] / df_trn_x["total_genre_count"].replace(0, np.nan)
).fillna(0)
df_tst_x["top_30_genre_ratio"] = (
    df_tst_x["top_30_genre_count"] / df_tst_x["total_genre_count"].replace(0, np.nan)
).fillna(0)

df_trn_x["top_30_50_genre_ratio"] = (
    df_trn_x["top_30_50_genre_count"] / df_trn_x["total_genre_count"].replace(0, np.nan)
).fillna(0)
df_tst_x["top_30_50_genre_ratio"] = (
    df_tst_x["top_30_50_genre_count"] / df_tst_x["total_genre_count"].replace(0, np.nan)
).fillna(0)

df_trn_x["top_50_70_genre_ratio"] = (
    df_trn_x["top_50_70_genre_count"] / df_trn_x["total_genre_count"].replace(0, np.nan)
).fillna(0)
df_tst_x["top_50_70_genre_ratio"] = (
    df_tst_x["top_50_70_genre_count"] / df_tst_x["total_genre_count"].replace(0, np.nan)
).fillna(0)

df_trn_x["bottom_30_genre_ratio"] = (
    df_trn_x["bottom_30_genre_count"] / df_trn_x["total_genre_count"].replace(0, np.nan)
).fillna(0)
df_tst_x["bottom_30_genre_ratio"] = (
    df_tst_x["bottom_30_genre_count"] / df_tst_x["total_genre_count"].replace(0, np.nan)
).fillna(0)

In [460]:
new_genre_features = [
    "top_30_genre_count",
    "top_30_50_genre_count",
    "top_50_70_genre_count",
    "bottom_30_genre_count",
    "total_genre_count",
    "top_30_genre_ratio",
    "top_30_50_genre_ratio",
    "top_50_70_genre_ratio",
    "bottom_30_genre_ratio"
]

In [461]:
df_trn_x.drop(columns=["top_30_genre_count",
    "top_30_50_genre_count",
    "top_50_70_genre_count",
    "bottom_30_genre_count",
    "total_genre_count"], inplace=True)
df_tst_x.drop(columns=["top_30_genre_count",
    "top_30_50_genre_count",
    "top_50_70_genre_count",
    "bottom_30_genre_count",
    "total_genre_count"], inplace=True)

In [463]:
df_trn_x.drop(columns=tag_cols, inplace=True)
df_tst_x.drop(columns=tag_cols, inplace=True)

In [464]:
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
import numpy as np

# 라벨을 통일해야 하는 컬럼 묶음
ks_color_cols = ["ks_color_1", "ks_color_2", "ks_color_3", "ks_color_4"]

adjective_cols = [
    "emotion_adjective_1", 
    "emotion_adjective_2", 
    "emotion_adjective_3", 
    "emotion_adjective_4"
]

# 원본 보존
df_trn_encoded_x = df_trn_x.copy()
df_tst_encoded_x = df_tst_x.copy()

In [465]:
# ks_color_1~4에 등장하는 모든 값을 하나로 모아서 fit
ks_values = df_trn_x[ks_color_cols].values.reshape(-1, 1)

ks_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

ks_encoder.fit(ks_values)

# train 변환
for col in ks_color_cols:
    df_trn_encoded_x[col] = ks_encoder.transform(df_trn_x[[col]])

# test 변환
for col in ks_color_cols:
    df_tst_encoded_x[col] = ks_encoder.transform(df_tst_x[[col]])

C:\Users\Dell5371\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but OrdinalEncoder was fitted without feature names
  warnings.warn(
C:\Users\Dell5371\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but OrdinalEncoder was fitted without feature names
  warnings.warn(
C:\Users\Dell5371\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but OrdinalEncoder was fitted without feature names
  warnings.warn(
C:\Users\Dell5371\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but OrdinalEncoder was fitted without feature names
  warnings.warn(
C:\Users\Dell5371\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but OrdinalEncoder was fitted without feature names
  warnings.warn(
C:\Users\Dell5371\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserW

# <b>Train model</b>

In [353]:
# =========================
# 최종 모델 학습 + test 평가
# =========================

final_model2 = XGBRegressor(
        random_state=42,
        objective="reg:squarederror",
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=5,
        gamma=0.1
    )

# 전체 train y 로그변환
df_trn_y_log = np.log1p(df_trn_y.values.ravel())

# 전체 train으로 학습
final_model2.fit(df_trn_encoded_x, df_trn_y_log)

# train / test 예측
y_train_pred_log = final_model2.predict(df_trn_encoded_x)
y_test_pred_log = final_model2.predict(df_tst_encoded_x)

# 원래 금액 단위 복원
y_train_pred = np.expm1(y_train_pred_log)
y_test_pred = np.expm1(y_test_pred_log)

# 음수 방지
y_train_pred = np.maximum(y_train_pred, 0)
y_test_pred = np.maximum(y_test_pred, 0)

# 실제값
y_train_true = df_trn_y.values.ravel()
y_test_true = df_tst_y.values.ravel()

# 최종 성능
final_result = pd.DataFrame([{
    "model": "XGBoost_final",
    
    "Train_MAE": mean_absolute_error(y_train_true, y_train_pred),
    "Test_MAE": mean_absolute_error(y_test_true, y_test_pred),
    
    "Train_MedianAE": median_absolute_error(y_train_true, y_train_pred),
    "Test_MedianAE": median_absolute_error(y_test_true, y_test_pred),
    
    "Train_RMSE": np.sqrt(mean_squared_error(y_train_true, y_train_pred)),
    "Test_RMSE": np.sqrt(mean_squared_error(y_test_true, y_test_pred)),
    
    "Train_R2": r2_score(y_train_true, y_train_pred),
    "Test_R2": r2_score(y_test_true, y_test_pred)
}])

final_result.round(4)

,model,Train_MAE,Test_MAE,Train_MedianAE,Test_MedianAE,Train_RMSE,Test_RMSE,Train_R2,Test_R2
0,XGBoost_final,20133.97,22250.93,4773.60,6663.77,40994.97,49416.46,0.90,0.85


# optuna

In [124]:
import time
import numpy as np
import pandas as pd
import optuna

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# =========================
# 전체 시작 시간
# =========================

total_start_time = time.time()

# =========================
# 1. X, y 준비
# =========================

prep_start_time = time.time()

X_train = df_trn_encoded_x.copy()
X_test = df_tst_encoded_x.copy()

y_train_true = df_trn_y.values.ravel()
y_test_true = df_tst_y.values.ravel()

# y 로그변환
y_train_log = np.log1p(y_train_true)

prep_end_time = time.time()
print(f"[1] 데이터 준비 시간: {prep_end_time - prep_start_time:.2f}초")

# =========================
# 2. train 내부에서 train/valid 분리
# =========================

split_start_time = time.time()

X_train_sub, X_valid, y_train_sub_log, y_valid_log, y_train_sub_true, y_valid_true = train_test_split(
    X_train,
    y_train_log,
    y_train_true,
    test_size=0.2,
    random_state=42
)

split_end_time = time.time()
print(f"[2] train/valid 분리 시간: {split_end_time - split_start_time:.2f}초")

# =========================
# 3. Optuna objective 함수
# =========================

def objective(trial):
    
    trial_start_time = time.time()
    
    params = {
        "random_state": 42,
        "objective": "reg:absoluteerror",
        "eval_metric": "mae",
        "tree_method": "hist",
        "n_jobs": -1,
        
        # 학습량
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        
        # 트리 복잡도
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        
        # 샘플링
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        
        # 정규화
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 3.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 30.0),
    }
    
    model = XGBRegressor(**params)
    
    model.fit(
        X_train_sub,
        y_train_sub_log,
        verbose=False
    )
    
    # validation 예측: 로그 스케일
    y_valid_pred_log = model.predict(X_valid)
    
    # 원래 금액 단위 복원
    y_valid_pred = np.expm1(y_valid_pred_log)
    y_valid_pred = np.maximum(y_valid_pred, 0)
    
    # 원래 금액 단위 MAE
    valid_mae = mean_absolute_error(y_valid_true, y_valid_pred)
    
    trial_end_time = time.time()
    
    print(
        f"Trial {trial.number} | "
        f"Valid MAE: {valid_mae:.4f} | "
        f"시간: {trial_end_time - trial_start_time:.2f}초"
    )
    
    return valid_mae

[1] 데이터 준비 시간: 0.00초
[2] train/valid 분리 시간: 0.00초


In [125]:
study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=50
)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-06-15 19:49:09,625] A new study created in memory with name: no-name-05e39fd4-4fb3-4c51-8260-8215fc51755f
[I 2026-06-15 19:49:11,796] Trial 0 finished with value: 44037.57920158267 and parameters: {'n_estimators': 667, 'learning_rate': 0.01028210013882417, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.9422102512012669, 'colsample_bytree': 0.611086023248821, 'gamma': 0.6817432790251585, 'reg_alpha': 1.2934979838536582, 'reg_lambda': 28.824763433084176}. Best is trial 0 with value: 44037.57920158267.


Trial 0 | Valid MAE: 44037.5792 | 시간: 2.17초


[I 2026-06-15 19:49:14,844] Trial 1 finished with value: 40464.80434310456 and parameters: {'n_estimators': 925, 'learning_rate': 0.05931115470976229, 'max_depth': 2, 'min_child_weight': 10, 'subsample': 0.8892081360801996, 'colsample_bytree': 0.7691717143809905, 'gamma': 0.31109397935163874, 'reg_alpha': 2.7332443966098667, 'reg_lambda': 18.074116199497862}. Best is trial 1 with value: 40464.80434310456.


Trial 1 | Valid MAE: 40464.8043 | 시간: 3.04초


[I 2026-06-15 19:49:18,802] Trial 2 finished with value: 43666.37544591133 and parameters: {'n_estimators': 756, 'learning_rate': 0.030392135755336214, 'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.9276859328628119, 'colsample_bytree': 0.8820899554537467, 'gamma': 0.5084437221638898, 'reg_alpha': 1.591255217295007, 'reg_lambda': 17.983957403575314}. Best is trial 1 with value: 40464.80434310456.


...

Trial 47 | Valid MAE: 39946.4104 | 시간: 5.20초


[I 2026-06-15 19:51:56,075] Trial 48 finished with value: 42102.095711059264 and parameters: {'n_estimators': 965, 'learning_rate': 0.04924045093167034, 'max_depth': 4, 'min_child_weight': 16, 'subsample': 0.6670223059969614, 'colsample_bytree': 0.7094730983638105, 'gamma': 0.08368731462372603, 'reg_alpha': 2.3567886821080974, 'reg_lambda': 17.069378782312214}. Best is trial 32 with value: 36038.26457479449.


Trial 48 | Valid MAE: 42102.0957 | 시간: 4.90초


[I 2026-06-15 19:51:59,273] Trial 49 finished with value: 41085.1232903314 and parameters: {'n_estimators': 756, 'learning_rate': 0.09039958879503071, 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.7030427357436209, 'colsample_bytree': 0.6867924192489668, 'gamma': 1.7250298567181688, 'reg_alpha': 2.2173369540523318, 'reg_lambda': 9.720700103997167}. Best is trial 32 with value: 36038.26457479449.


Trial 49 | Valid MAE: 41085.1233 | 시간: 3.19초
Best MAE: 36038.26457479449
Best params: {'n_estimators': 955, 'learning_rate': 0.08085052167943471, 'max_depth': 4, 'min_child_weight': 17, 'subsample': 0.7200247864446896, 'colsample_bytree': 0.7138505770668886, 'gamma': 0.3206907131215627, 'reg_alpha': 1.977447861369854, 'reg_lambda': 3.2939769332775097}


In [126]:
best_params = study.best_params.copy()

final_model = XGBRegressor(
    random_state=42,
    objective="reg:absoluteerror",
    eval_metric="mae",
    tree_method="hist",
    n_jobs=-1,
    **best_params
)

final_model.fit(
    X_train,
    y_train_log,
    verbose=False
)

,objective,'reg:absoluteerror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7138505770668886
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mae'


In [127]:
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    mean_squared_error,
    r2_score
)

y_train_pred_log = final_model.predict(X_train)
y_test_pred_log = final_model.predict(X_test)

# 원래 금액 단위 복원
y_train_pred = np.expm1(y_train_pred_log)
y_test_pred = np.expm1(y_test_pred_log)

# 음수 방지
y_train_pred = np.maximum(y_train_pred, 0)
y_test_pred = np.maximum(y_test_pred, 0)

train_r2 = r2_score(y_train_true, y_train_pred)
test_r2 = r2_score(y_test_true, y_test_pred)

final_result = pd.DataFrame([{
    "model": "XGBoost_Optuna_no_KFold",
    
    "Train_MAE": mean_absolute_error(y_train_true, y_train_pred),
    "Test_MAE": mean_absolute_error(y_test_true, y_test_pred),
    
    "Train_MedianAE": median_absolute_error(y_train_true, y_train_pred),
    "Test_MedianAE": median_absolute_error(y_test_true, y_test_pred),
    
    "Train_RMSE": np.sqrt(mean_squared_error(y_train_true, y_train_pred)),
    "Test_RMSE": np.sqrt(mean_squared_error(y_test_true, y_test_pred)),
    
    "Train_R2": train_r2,
    "Test_R2": test_r2,
    "R2_gap": train_r2 - test_r2
}])

final_result.round(4)

,model,Train_MAE,Test_MAE,Train_MedianAE,Test_MedianAE,Train_RMSE,Test_RMSE,Train_R2,Test_R2,R2_gap
0,XGBoost_Optuna_no_KFold,9425.03,19750.69,243.71,6707.42,27273.46,41900.75,0.96,0.89,0.06


In [129]:
# =========================
# 9. Optuna 결과 상위 10개
# =========================

trials_df = study.trials_dataframe()

trials_view = trials_df.sort_values("value", ascending=True)

trials_view[[
    "number",
    "value",
    "params_n_estimators",
    "params_learning_rate",
    "params_max_depth",
    "params_min_child_weight",
    "params_subsample",
    "params_colsample_bytree",
    "params_gamma",
    "params_reg_alpha",
    "params_reg_lambda"
]].head(10)

,number,value,params_n_estimators,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,params_colsample_bytree,params_gamma,params_reg_alpha,params_reg_lambda
32,32,36038.26,955,0.08,4,17,0.72,0.71,0.32,1.98,3.29
11,11,37469.24,985,0.10,2,15,1.00,0.74,0.02,2.21,6.86
22,22,37618.16,741,0.08,3,17,0.71,0.65,0.15,1.88,13.56
33,33,38415.23,945,0.06,4,18,0.71,0.60,0.36,2.77,3.68
17,17,38549.87,720,0.03,2,13,0.83,0.94,0.20,2.35,22.31
13,13,38577.56,830,0.10,5,15,0.68,0.71,0.16,2.12,6.59
14,14,38773.38,888,0.10,2,14,0.98,0.69,0.28,2.14,5.14
31,31,38970.34,844,0.10,5,15,0.70,0.70,0.16,2.14,6.54
18,18,39167.65,706,0.02,2,12,0.82,0.95,0.50,2.53,24.13
16,16,39188.66,991,0.05,3,20,0.68,0.66,0.68,2.00,4.52


In [130]:
trials_view[[
    "number",
    "value",
    "params_n_estimators",
    "params_learning_rate",
    "params_max_depth",
    "params_min_child_weight",
    "params_subsample",
    "params_colsample_bytree",
    "params_gamma",
    "params_reg_alpha",
    "params_reg_lambda"
]].head(10).to_csv('params1.csv',encoding='utf-8-sig')